# Final Model Improvement Experiments

This notebook investigates whether the predictive performance of the
selected forest-fire burned-area model can be improved further.

The current final model is a weighted ensemble consisting of:

- 70% HistGradientBoosting Regressor
- 30% Random Forest Regressor

The baseline model achieved:

- Test R²: 0.2302
- Test MAE: 0.8328
- Test RMSE: 1.0421

Although this model provides the strongest performance obtained so far,
previous error analysis showed substantial underprediction of rare
extreme-fire events.

Therefore, this notebook investigates alternative modelling strategies,
including advanced gradient boosting, alternative target transformations,
cross-validation, and severity-aware weighting.

The existing final model is retained as the baseline and will not be
modified.

The objective is to determine whether a statistically and scientifically
meaningful improvement can be achieved without introducing data leakage
or excessive model complexity.

In [1]:
# IMPORT LIBRARIES

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor
)

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# LOAD PROCESSED DATA

PROJECT_ROOT = Path("../../")
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"

X_train = pd.read_csv(
    PROCESSED_DIR / "X_train.csv"
)

X_test = pd.read_csv(
    PROCESSED_DIR / "X_test.csv"
)

y_train = pd.read_csv(
    PROCESSED_DIR / "y_train.csv"
).squeeze("columns")

y_test = pd.read_csv(
    PROCESSED_DIR / "y_test.csv"
).squeeze("columns")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (9042, 44)
X_test : (2261, 44)
y_train: (9042,)
y_test : (2261,)


In [3]:
# LOAD CURRENT BEST MODEL

MODEL_DIR = PROJECT_ROOT / "models"

baseline_hist = joblib.load(
    MODEL_DIR / "hist_gradient_boosting_final.pkl"
)

baseline_rf = joblib.load(
    MODEL_DIR / "random_forest_final.pkl"
)

baseline_hist_pred = baseline_hist.predict(X_test)
baseline_rf_pred = baseline_rf.predict(X_test)

baseline_pred = (
    0.70 * baseline_hist_pred
    + 0.30 * baseline_rf_pred
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_pred
    )
)

print("=" * 55)
print("CURRENT BASELINE MODEL")
print("=" * 55)

print(f"R²   : {baseline_r2:.4f}")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")

CURRENT BASELINE MODEL
R²   : 0.2302
MAE  : 0.8328
RMSE : 1.0421


## XGBoost

In [5]:
try:
    import xgboost as xgb

    print("XGBoost version:", xgb.__version__)
    print("XGBoost is available.")

except ImportError:
    print("XGBoost is NOT installed in the current environment.")

XGBoost version: 3.4.0
XGBoost is available.


In [6]:
# XGBOOST BASELINE MODEL

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.0


In [7]:
# TRAIN XGBOOST BASELINE

xgb_baseline = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")

xgb_baseline.fit(
    X_train,
    y_train
)

print("XGBoost training completed.")

Training XGBoost...
XGBoost training completed.


In [8]:
# EVALUATE XGBOOST


xgb_train_pred = xgb_baseline.predict(X_train)
xgb_test_pred = xgb_baseline.predict(X_test)

xgb_train_r2 = r2_score(
    y_train,
    xgb_train_pred
)

xgb_test_r2 = r2_score(
    y_test,
    xgb_test_pred
)

xgb_test_mae = mean_absolute_error(
    y_test,
    xgb_test_pred
)

xgb_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_test_pred
    )
)

print("=" * 55)
print("XGBOOST BASELINE RESULTS")
print("=" * 55)

print(f"Train R² : {xgb_train_r2:.4f}")
print(f"Test R²  : {xgb_test_r2:.4f}")
print(f"Test MAE : {xgb_test_mae:.4f}")
print(f"Test RMSE: {xgb_test_rmse:.4f}")

print()
print("CURRENT ENSEMBLE BASELINE")
print("=" * 55)

print(f"Test R²  : {baseline_r2:.4f}")
print(f"Test MAE : {baseline_mae:.4f}")
print(f"Test RMSE: {baseline_rmse:.4f}")

XGBOOST BASELINE RESULTS
Train R² : 0.6931
Test R²  : 0.2356
Test MAE : 0.8284
Test RMSE: 1.0385

CURRENT ENSEMBLE BASELINE
Test R²  : 0.2302
Test MAE : 0.8328
Test RMSE: 1.0421
